In [129]:
_SKILLS_SYSTEM_PROMPT = """
    You are an expert technical recruiter and organizational psychologist specializing in behavioral competency frameworks.
    Extract ALL hard and soft skills from the provided text and assign each a relevance weight according to importance signals.

    ## Step 1 — Read responsibilities first
    Before extracting skills, identify the core responsibilities. Use them as context for ALL skill descriptions,
    especially tools listed by name without usage detail. Never use a tool's generic textbook purpose as context.

    ## Step 2 — Proficiency vocabulary
    Every hard skill description MUST begin with one of these phrases:

    | Signal in text                                      | Use                        |
    |-----------------------------------------------------|----------------------------|
    | "exposure to", "nice to have", implied              | Familiar with              |
    | "knowledge of", "preferred", "diferencial"          | Working knowledge of       |
    | "experience with/in", mentioned without qualifier   | Experience with            |
    | "proficient", "solid", required technical skill     | Proficient in              |
    | "lead", "design", "architect", deep ownership       | Expert-level mastery of    |

    Choose the phrase that fits the subject: `Proficient in Python`, `Experience with microservices architectures`,
    `Working knowledge of Kafka` — not `Hands-on use of microservices architectures`.

    ## Step 3 — Hard skills
    Tools, technologies, methodologies, and domain knowledge.

    Format: `<proficiency phrase> <canonical name> [for <context from responsibilities>]`

    The context clause is required when the tool's role in this job differs from its common use.
    Skip it when it would be redundant (e.g. `Proficient in Python` needs no clause).

    Rules:
    1. Extract the broad category AND each specific tool separately, each with its own proficiency and context.
    2. Canonical names only: `SQL`, `Apache Kafka`, `Azure Data Factory` — not too broad, not too version-specific.
    3. Language requirements are hard skills: e.g. `Proficient in English`.
    4. `time_experience_months`: populate only if an explicit duration is stated; otherwise null.
    5. "Preferred" / "diferencial" tools: drop one proficiency tier and reduce weight vs. required equivalents.

    ## Step 4 — Soft skills
    Interpersonal, behavioral, or cognitive traits — never tools, domain knowledge, or language proficiency.

    Format: `<verb>-ing <object> [<context>]`
    Examples: `Collaborating across engineering and product teams`, `Escalating risks before they impact delivery`

    Use verb phrases only — not nouns like "Stakeholder Communication".
    Split sentences that imply multiple distinct traits into separate skills.

    ## Weight assignment

    | Proficiency level              | Required/emphasized | Preferred | Implied  |
    |--------------------------------|---------------------|-----------|----------|
    | Expert-level mastery of        | 0.90 – 1.0          | —         | —        |
    | Proficient in                  | 0.75 – 0.90         | 0.55–0.70 | —        |
    | Experience with                | 0.55 – 0.75         | 0.35–0.55 | —        |
    | Working knowledge of           | 0.30 – 0.55         | 0.20–0.35 | —        |
    | Familiar with                  | —                   | 0.15–0.30 | 0.10–0.20|

    All output must be in English, even if the source text is in another language.
"""

### Market Utils

In [136]:
import re
import time
import pandas as pd
import psycopg2
from os import getenv
from typing import Optional
from google import genai
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from concurrent.futures import ThreadPoolExecutor, as_completed
from pgvector.psycopg2 import register_vector
from pydantic import BaseModel, Field

MODEL_NAME = "gemini-2.5-flash-lite"
EMBEDDING_MODEL = "gemini-embedding-001"
LLM_PROVIDER = "google_genai"
LLM_TEMPERATURE = 0.6
DB_HOST = "localhost"
DB_NAME = "market_fit"
DB_USER = getenv("DB_USER")
DB_PASSWORD = getenv("DB_PASSWORD")
GEMINI_API_KEY = getenv("GEMINI_API_KEY")
JOB_POSTINGS_TABLE = "job_postings"
HARD_SKILLS_TABLE = "hard_skills"
SOFT_SKILLS_TABLE = "soft_skills"

SKILLS_CONCURRENCY = 10
EMBED_CONCURRENCY = 5
EMBED_BATCH_SIZE = 100
EMBED_MAX_RETRIES = 5

# ---------------------------------------------------------------------------

class HardSkill(BaseModel):
    description: str = Field(description="Objective description of the hard skill or experience in English.")
    time_experience: Optional[float] = Field(description="Experience in months, if explicitly stated.")
    weight: Optional[float] = Field(description="Relevance score 0–1 for this position.")

class SoftSkill(BaseModel):
    description: str = Field(description="Canonical name of the soft skill in English (e.g. 'Stakeholder Communication').")
    weight: Optional[float] = Field(description="Relevance score 0–1 for this position.")

class SkillsList(BaseModel):
    hard_skills: list[HardSkill]
    soft_skills: list[SoftSkill]

def extract_skills_bulk(
    chat_model,
    df: pd.DataFrame,
    concurrency: int = SKILLS_CONCURRENCY,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Extract hard and soft skills for every job in df.
    Returns (hard_skills_df, soft_skills_df).
    """

    def _extract_skills_for_row(chain, row: dict) -> tuple[str, SkillsList]:
        text = f"Job title: {row['title']}\nJob description: {row['description']}"
        try:
            skills = chain.invoke({"input": text})
        except Exception as exc:
            print(f"Skills extraction failed for job {row['id']}: {exc}")
            skills = SkillsList(hard_skills=[], soft_skills=[])
        return row["id"], skills

    prompt = ChatPromptTemplate.from_messages([
        ("system", _SKILLS_SYSTEM_PROMPT),
        ("human", "{input}"),
    ])
    chain = prompt | chat_model.with_structured_output(schema=SkillsList)
    rows = df.to_dict(orient="records")

    job_results: list[tuple[str, SkillsList]] = [None] * len(rows)  # type: ignore[list-item]
    with ThreadPoolExecutor(max_workers=concurrency) as pool:
        futures = {pool.submit(_extract_skills_for_row, chain, row): i for i, row in enumerate(rows)}
        for future in as_completed(futures):
            job_results[futures[future]] = future.result()

    hard_records, soft_records = [], []
    for job_id, skills in job_results:
        for skill in skills.hard_skills:
            hard_records.append({
                "job_id": job_id,
                "skill_description": skill.description,
                "time_experience": skill.time_experience,
                "weight": skill.weight,
            })
        for skill in skills.soft_skills:
            soft_records.append({
                "job_id": job_id,
                "skill_description": skill.description,
                "weight": skill.weight,
            })

    return pd.DataFrame(hard_records), pd.DataFrame(soft_records)

def embed_column(
    client: genai.Client,
    texts: list[str],
    batch_size: int = EMBED_BATCH_SIZE,
    concurrency: int = EMBED_CONCURRENCY,
) -> list[list[float]]:
    """Embed an arbitrary list of strings in parallel batches, preserving order."""

    def _embed_batch_with_retry(
        client: genai.Client,
        texts: list[str],
        max_retries: int = EMBED_MAX_RETRIES,
    ) -> list[list[float]]:
        for attempt in range(max_retries):
            try:
                response = client.models.embed_content(model=EMBEDDING_MODEL, contents=texts)
                return [e.values for e in response.embeddings]
            except genai.errors.ClientError as exc:
                is_rate_limit = exc.code == 429 or "RESOURCE_EXHAUSTED" in str(exc.status)
                if not is_rate_limit or attempt == max_retries - 1:
                    raise
                match = re.search(r"retry in (\d+(?:\.\d+)?)s", str(exc))
                wait = float(match.group(1)) if match else 2 ** attempt * 10
                print(f"Rate limited — waiting {wait:.0f}s (attempt {attempt + 1}/{max_retries})")
                time.sleep(wait)
        raise RuntimeError("Max retries exceeded")

    batches = [texts[i : i + batch_size] for i in range(0, len(texts), batch_size)]
    results: list[list[list[float]]] = [None] * len(batches)  # type: ignore[list-item]

    with ThreadPoolExecutor(max_workers=concurrency) as pool:
        futures = {pool.submit(_embed_batch_with_retry, client, b): i for i, b in enumerate(batches)}
        for future in as_completed(futures):
            results[futures[future]] = future.result()

    return [emb for batch in results for emb in batch]

def db_connect() -> psycopg2.extensions.connection:
    conn = psycopg2.connect(
        host=DB_HOST, dbname=DB_NAME, user=DB_USER, password=DB_PASSWORD
    )
    register_vector(conn)
    return conn

def load_job_postings() -> pd.DataFrame:
    with db_connect() as conn:
        with conn.cursor() as cur:
            cur.execute(f"SELECT id, title, description FROM {JOB_POSTINGS_TABLE}")
            rows = cur.fetchall()
            cols = [desc[0] for desc in cur.description]
    return pd.DataFrame(rows, columns=cols)

In [131]:
# Market Logic
chat_model = init_chat_model(
    model=MODEL_NAME,
    model_provider=LLM_PROVIDER,
    temperature=LLM_TEMPERATURE,
    api_key=GEMINI_API_KEY,
)

jobs_df = load_job_postings()
print(f"{len(jobs_df)} postings loaded.")

hard_df, soft_df = extract_skills_bulk(chat_model, jobs_df)
print(f"{len(hard_df)} hard skills / {len(soft_df)} soft skills extracted.")
hard_df

298 postings loaded.
6535 hard skills / 4603 soft skills extracted.


,job_id,skill_description,time_experience,weight
0,2124334619,Proficient in Scala,NaN,0.85
1,2124334619,Proficient in Spark Streaming,NaN,0.85
2,2124334619,Proficient in Python,NaN,0.85
3,2124334619,Experience with distributed architectures,NaN,0.65
4,2124334619,Experience with microservices architectures,NaN,0.65
...,...,...,...,...
6530,2068696904,Working knowledge of SAP,NaN,0.35
6531,2068696904,Working knowledge of enterprise data systems,NaN,0.35
6532,2068696904,Familiar with data security,NaN,0.25
6533,2068696904,Familiar with compliance frameworks,NaN,0.25


### Resume Utils

In [135]:
import os
import numpy as np
import pandas as pd
from psycopg2 import pool
from contextlib import contextmanager
from sklearn.cluster import DBSCAN
from pgvector.psycopg2 import register_vector
from typing import List
from os import getenv
from psycopg2.extras import execute_values
from psycopg2.extensions import connection as PsycopgConnection
import uuid

SOFT_SKILLS_SIMILARITY_THRESHOLD = 0.66
HARD_SKILLS_SIMILARITY_THRESHOLD = 0.66
SOFT_SKILLS_WEIGHT_COLUMN_INDEX = 3
SOFT_SKILLS_STRING_COLUMN_INDEX = 4
HARD_SKILLS_STRING_COLUMN_INDEX = 5
HARD_SKILLS_WEIGHT_COLUMN_INDEX = 3

JOB_POSTINGS_TABLE = "job_postings"
HARD_SKILLS_TABLE = "hard_skills"
SOFT_SKILLS_TABLE = "soft_skills"
RESUMES_TABLE = "resumes"
CANDIDATE_HARD_SKILLS_TABLE = "candidate_hard_skills"
CANDIDATE_SOFT_SKILLS_TABLE = "candidate_soft_skills"

LLM_MODEL_VECTOR_DIMENSIONS = 3072
EMBEDDING_MODEL = "gemini-embedding-001"
LLM_MODEL_NAME = "gemini-2.5-flash-lite"
LLM_PROVIDER = "google_genai"
LLM_TEMPERATURE = 0.5
EMBED_BATCH_SIZE = 100
EMBED_CONCURRENCY = 5
EMBED_MAX_RETRIES = 5

DB_NAME = "market_fit"
DB_HOST = "localhost"
db_user = os.getenv("DB_USER")
db_pw = os.getenv("DB_PASSWORD")
api_key = getenv('GEMINI_API_KEY')

SENIORITY_LEVELS = (
    "Intern", "Junior", "Mid", "Senior", "Associate", "Specialist",
    "Manager", "Director", "Head", "President/Vice President",
    "C-Level", "Partner", "Owner", "Founder",
)


class DatabaseManager:
    def __init__(self):
        self.db_config = {
            "dbname": DB_NAME,
            "host": DB_HOST,
            "user": db_user,
            "password": db_pw
        }
        self._pool = None

    @property
    def db_pool(self):
        """Lazy initialization: The pool is only created when first accessed."""
        if self._pool is None:
            print("Initializing connection pool...")
            self._pool = pool.SimpleConnectionPool(
                minconn=1,
                maxconn=10,
                **self.db_config
            )
        return self._pool

    @contextmanager
    def get_conn(self):
        """Context manager to handle connection lifecycle."""
        conn = self.db_pool.getconn()
        try:
            register_vector(conn)
            yield conn
        finally:
            self.db_pool.putconn(conn)

    def get_resume(self, resume_id: str) -> pd.DataFrame:
        query = f"SELECT * FROM {RESUMES_TABLE} WHERE id = %s"
        with self.get_conn() as conn:
            with conn.cursor() as cur:
                cur.execute(query, (resume_id,))
                rows = cur.fetchall()
                cols = [desc[0] for desc in cur.description]
        return pd.DataFrame(rows, columns=cols)

    def filter_job_postings(self, industries: list) -> pd.DataFrame:
        query = f"SELECT id, title, description FROM {JOB_POSTINGS_TABLE} WHERE ai_industries::TEXT[] && %s::TEXT[]"
        with self.get_conn() as conn:
            with conn.cursor() as cur:
                cur.execute(query, (industries,))
                rows = cur.fetchall()
                cols = [desc[0] for desc in cur.description]
        return pd.DataFrame(rows, columns=cols)

    def get_position_skills(self, jobs_ids: list, table_name: str) -> pd.DataFrame:
        query = f"SELECT * FROM {table_name} WHERE job_id = ANY(%s)"
        with self.get_conn() as conn:
            with conn.cursor() as cur:
                cur.execute(query, (jobs_ids,))
                rows = cur.fetchall()
                cols = [desc[0] for desc in cur.description]
        return pd.DataFrame(rows, columns=cols)

    def get_candidate_skills(self, resume_id: str, table_name: str) -> pd.DataFrame:
        query = f"SELECT * FROM {table_name} WHERE resume_id = %s"
        with self.get_conn() as conn:
            with conn.cursor() as cur:
                cur.execute(query, (resume_id,))
                rows = cur.fetchall()
                cols = [desc[0] for desc in cur.description]
        return pd.DataFrame(rows, columns=cols)
    
    def close_all(self):
        """Cleanly shut down the pool when the app stops."""
        if self._pool:
            self._pool.closeall()

class MarketSkillsMatrix:
    def __init__(self, database_manager: DatabaseManager, skill_type: str, candidate_industries: list):
        if skill_type not in ("hard", "soft"):
            raise ValueError(f"skill_type must be 'hard' or 'soft', got '{skill_type}'")
        if candidate_industries is None:
            raise ValueError("jobs_df cannot be empty")

        self.db = database_manager
        self.skill_type = skill_type
        self.candidate_industries = candidate_industries

        # Populated by _initialize_matrices
        self.string_matrix: np.ndarray = None      # (n_jobs, max_skills) skill descriptions
        self.weight_matrix: np.ndarray = None      # (n_jobs, max_skills) skill weights
        self.embedding_matrix: np.ndarray = None   # (n_jobs, max_skills, vector_dim) embeddings
        self.skills_count_by_index: list[int] = [] # count of skills per job index 
        self.job_id_by_index: list = []            # job_id mapped to matrix row index

        # Populated after combine() / weight_against() calls
        self._match_score_matrix: np.ndarray = None     # raw match counts per (job, skill) cell, starts with 0s and 1s
        self._weighted_match_matrix: np.ndarray = None  # weight-qualified match counts

        self._initialize_matrices()

    def _initialize_matrices(self) -> None:
        jobs_df = self.db.filter_job_postings(self.candidate_industries)
        matching_jobs_ids = jobs_df["id"].tolist()
        table = SOFT_SKILLS_TABLE if self.skill_type == "soft" else HARD_SKILLS_TABLE
        skills_df = self.db.get_position_skills(matching_jobs_ids, table).sort_values("job_id")

        skills_df["index"] = skills_df.groupby("job_id").ngroup()
        max_skills_per_job = skills_df.groupby("index").size().max()

        self.string_matrix = build_padded_matrix(
            skills_df, "skill_description", max_skills_per_job, pad_value=""
        )
        self.weight_matrix = build_padded_matrix(
            skills_df, "weight", max_skills_per_job, pad_value=0, dtype=np.float32
        )
        self.embedding_matrix = build_embedding_matrix(
            skills_df, max_skills_per_job
        )
        self.skills_count_by_index = (
            skills_df["index"].value_counts().sort_index().tolist()
        )
        self.job_id_by_index = (
            skills_df[["index", "job_id"]].drop_duplicates()["job_id"].tolist()
        )
        self._match_score_matrix = create_match_score_matrix(self.skills_count_by_index)
        self._weighted_match_matrix = np.zeros_like(self._match_score_matrix)

    def accumulate_matches(self, match_array: np.ndarray) -> None:
        """Add a match score array into the running match score matrix."""
        if match_array.shape != self._match_score_matrix.shape:
            raise ValueError(
                f"match_array shape {match_array.shape} does not match "
                f"expected {self._match_score_matrix.shape}"
            )
        self._match_score_matrix += match_array

    def accumulate_weighted_matches(
        self, candidate_skill_weight: float, binary_mask: np.ndarray
    ) -> None:
        """Record which job skills are met by a candidate skill at the given weight."""
        if binary_mask.shape != self.weight_matrix.shape:
            raise ValueError(
                f"binary_mask shape {binary_mask.shape} does not match "
                f"weight_matrix shape {self.weight_matrix.shape}"
            )
        candidate_weight_mask = binary_mask * candidate_skill_weight
        weight_qualified = (candidate_weight_mask >= self.weight_matrix) & (self.weight_matrix != 0)
        self._weighted_match_matrix += weight_qualified.astype(np.int8)

    def get_min_compliance_pct_by_job(self) -> list[float]:
        """
        Percentage of each job's skills matched by the candidate, not considering weight.
        """
        qualifying = (self._match_score_matrix > 1).sum(axis=1)  # 0 means padding and n > 1 means a candidate skill matched that job skill n times
        return [round(100 * a / b, 2) for a, b in zip(qualifying.tolist(), self.skills_count_by_index)]

    def get_ideal_compliance_pct_by_job(self) -> list[float]:
        """
        Percentage of each job's skills met at or above their required weight by candidate.
        """
        qualifying = (self._weighted_match_matrix != 0).sum(axis=1)
        return [round(100 * a / b, 2) for a, b in zip(qualifying.tolist(), self.skills_count_by_index)]

    def get_matched_skills(self, job_index: int, top_n: int = 5, with_scores: bool = False):
        scores = self._match_score_matrix[job_index]
        descriptions = self.string_matrix[job_index]
        ranked = sorted(
            ((desc, int(score)) for desc, score in zip(descriptions, scores) if desc != "" and score > 0),
            key=lambda x: x[1],
            reverse=True,
        )
        return ranked if with_scores else [desc for desc, _ in ranked]

class StructuredOutputParsingError(Exception):
    def __init__(self, detail: str = "Failed to parse structured output from LLM"):
        self.detail = detail
        super().__init__(detail)


def _safe_scalar(value):
    """Convert a single value to a safe type for database insertion."""
    if hasattr(value, "item"):
        value = value.item()
    if isinstance(value, float) and value != value:
        return 0
    if isinstance(value, int) and not isinstance(value, bool):
        if value < -32768 or value > 32767:
            return 0
    return value

def _insert_df(conn: PsycopgConnection, df: pd.DataFrame, table_name: str) -> None:
    """Low-level insert — expects an already-open connection, no commit."""
    cols = ", ".join(df.columns)
    placeholders = ", ".join(
        "%s::vector" if "embedding" in col else "%s"
        for col in df.columns
    )
    template = f"({placeholders})"
    cur = conn.cursor()
    execute_values(
        cur,
        f"INSERT INTO {table_name} ({cols}) VALUES %s",
        [tuple(_safe_scalar(x) for x in row) for _, row in df.iterrows()],
        template=template,
        page_size=500
    )

def get_static_list_of_industries() -> List[str]:
    """Load a static list of industries from a text file."""
    with open("../data/enum_industry.txt", "r") as f:
        industries = [line.strip() for line in f if line.strip()]
        industries = [industry.upper() for industry in industries]
    return industries

def save_resume_data(
    conn: PsycopgConnection,
    cv_df: pd.DataFrame,
    hard_skills_df: pd.DataFrame,
    soft_skills_df: pd.DataFrame,
) -> None:
    """Insert resume + skills atomically — all succeed or all roll back."""
    try:
        _insert_df(conn, cv_df, RESUMES_TABLE)
        print(f"Inserted to {RESUMES_TABLE}")
        _insert_df(conn, hard_skills_df, CANDIDATE_HARD_SKILLS_TABLE)
        print(f"Inserted to {CANDIDATE_HARD_SKILLS_TABLE}")
        _insert_df(conn, soft_skills_df, CANDIDATE_SOFT_SKILLS_TABLE)
        print(f"Inserted to {CANDIDATE_SOFT_SKILLS_TABLE}")
        conn.commit()
    except Exception:
        conn.rollback()
        raise

def cosine_similarities_matrix(query: np.ndarray, matrix: np.ndarray) -> np.ndarray:
    zero_mask = np.all(matrix == 0, axis=-1)  
    query_norm = np.linalg.norm(query)
    row_norms = np.linalg.norm(matrix, axis=-1)  
    dot_products = matrix @ query
    similarities = dot_products / (row_norms * query_norm + 1e-10)
    similarities[zero_mask] = 0.0
    return similarities  

def create_match_score_matrix(count_list: list):
    nrows = len(count_list)
    ncols = max(count_list)
    matrix = np.zeros((nrows, ncols), dtype=np.int8)
    for i, count in enumerate(count_list):
        matrix[i, :count] = 1
    return matrix     

def analyze_market(market_obj: MarketSkillsMatrix, candidate_skills_df: pd.DataFrame, skills_type: str) -> None:
    weight_column_index = SOFT_SKILLS_WEIGHT_COLUMN_INDEX if skills_type == "soft" else HARD_SKILLS_WEIGHT_COLUMN_INDEX
    string_column_index = SOFT_SKILLS_STRING_COLUMN_INDEX if skills_type == "soft" else HARD_SKILLS_STRING_COLUMN_INDEX
    threshold = SOFT_SKILLS_SIMILARITY_THRESHOLD if skills_type == "soft" else HARD_SKILLS_SIMILARITY_THRESHOLD
    skills_count = candidate_skills_df.shape[0]

    for i in range(0, skills_count):
        weight = candidate_skills_df.iloc[(i, weight_column_index)] 
        skill_embedding = candidate_skills_df.iloc[(i, string_column_index) ] 
        cosine_similarities = cosine_similarities_matrix(skill_embedding, market_obj.embedding_matrix)
        binary_mask = (cosine_similarities > threshold).astype(np.int8)
        market_obj.accumulate_matches(binary_mask)
        market_obj.accumulate_weighted_matches(weight, binary_mask)

def build_padded_matrix(
        df: pd.DataFrame,
        column_name: str,
        length: int,
        pad_value,
        dtype=None,
    ) -> np.ndarray:
        rows = [
            np.pad(
                array=group[column_name].values,
                pad_width=(0, length - len(group)),
                constant_values=pad_value,
            )
            for _, group in df.groupby("index")
        ]
        return np.array(rows, dtype=dtype)

def build_embedding_matrix(
    df: pd.DataFrame, max_skills: int
    ) -> np.ndarray:
        zero_vector = np.zeros(LLM_MODEL_VECTOR_DIMENSIONS)
        rows = [
            np.vstack(
                list(group["embedding"].values)
                + [zero_vector] * (max_skills - len(group))
            )
            for _, group in df.groupby("index")
        ]
        return np.array(rows, dtype=np.float32)

def get_market_analysis_results(market_obj: MarketSkillsMatrix) -> list[dict]:
    compliance_by_job = market_obj.get_min_compliance_pct_by_job()
    ideal_compliance_by_job = market_obj.get_ideal_compliance_pct_by_job()
    noncompliance_mask = market_obj._match_score_matrix == 1

    return [
        {
            "job_index": i,
            "job_id": job_id,
            "minimum_compliance_pct": compliance_pct,
            "ideal_compliance_pct": ideal_compliance_pct,

            "nonmatched_skills_count": int(noncompliance_mask[i].sum()),
            "nonmatched_skills": list(set(market_obj.string_matrix[i][noncompliance_mask[i]])),
            
            "matched_skills": list(
                market_obj.string_matrix[i][
                    (market_obj._match_score_matrix[i] > 1) &
                    (market_obj.string_matrix[i] != "")
                ]
            ),
            "similarity_match_scores": market_obj.get_matched_skills(i, with_scores=True),

            "not_ideal_skills": list(set(
                    market_obj.string_matrix[i][
                        (market_obj._weighted_match_matrix[i] == 0) &
                        (market_obj.string_matrix[i] != "")
                    ]
                ))
        }
        for i, (job_id, compliance_pct, ideal_compliance_pct) in enumerate(
            zip(market_obj.job_id_by_index, compliance_by_job, ideal_compliance_by_job)
        )
    ]

def build_analysis_display(market_obj: MarketSkillsMatrix, analysis: list[dict]) -> pd.DataFrame:
    sorted_analysis = sorted(analysis, key=lambda e: e["minimum_compliance_pct"], reverse=True)
    sorted_counts = [market_obj.skills_count_by_index[market_obj.job_id_by_index.index(e["job_id"])] for e in sorted_analysis]

    df = pd.DataFrame([
        {
            "job_id": entry["job_id"],
            "job_index": entry["job_index"],
            "required_skills": count,
            "matched_count": len(entry["matched_skills"]),
            "minimum_compliance_pct": entry["minimum_compliance_pct"],
            "matched_skills": entry["matched_skills"],

            "insufficient_count": len(entry["not_ideal_skills"]),
            "ideal_compliance_pct": entry["ideal_compliance_pct"],
            "insufficient_proficiency": entry["not_ideal_skills"],

            "nonmatched_count": entry["nonmatched_skills_count"],
            "nonmatched_skills": entry["nonmatched_skills"],
        }
        for entry, count in zip(sorted_analysis, sorted_counts)
    ])
    return df

def matches_display(market_obj: MarketSkillsMatrix, analysis: list[dict]) -> pd.DataFrame:
    flat_data = [
        (skill, score, entry["job_index"])
        for entry in analysis
        for skill, score in entry["similarity_match_scores"]
    ]
    if not flat_data:
        return pd.DataFrame()

    matches_df = pd.DataFrame(flat_data, columns=["skill", "score", "job_index"])

    skill_stats = matches_df.groupby("skill").agg(
        total_matches=("score", "sum"),
        job_indices=("job_index", set)
    ).reset_index()
    flat_strings = market_obj.string_matrix.flatten()
    flat_embeddings = market_obj.embedding_matrix.reshape(-1, LLM_MODEL_VECTOR_DIMENSIONS)
    
    skill_stats["embedding"] = [
        flat_embeddings[np.where(flat_strings == skill)[0][0]]
        if np.where(flat_strings == skill)[0].size > 0 else None
        for skill in skill_stats["skill"]
    ]
    skill_stats = skill_stats.dropna(subset=["embedding"])

    embeddings = np.vstack(skill_stats["embedding"].values)
    skill_stats["cluster"] = DBSCAN(eps=0.15, min_samples=1, metric="cosine").fit_predict(embeddings)

    total_jobs = len(analysis)
    df = (
        skill_stats
        .groupby("cluster")
        .agg(
            skill_variants=("skill", list),
            total_matches=("total_matches", "sum"),
            unique_jobs=("job_indices", lambda sets: len(set.union(*sets)))
        )
        .assign(job_coverage_pct=lambda df: (df["unique_jobs"] / total_jobs * 100).round(2))
        .sort_values("total_matches", ascending=False)
        .reset_index(drop=True)  # drops the cluster index cleanly
    )
    return df[df["skill_variants"].apply(len) < 10] # Avoids poorly formed skill variants where a large number of non related items are grouped together

def nonmatches_display(market_obj: MarketSkillsMatrix) -> pd.DataFrame:
    nonmatch_mask = market_obj._match_score_matrix == 1
    flat_embeddings = market_obj.embedding_matrix[nonmatch_mask]
    flat_descriptions = market_obj.string_matrix[nonmatch_mask]

    if len(flat_embeddings) == 0:
        return pd.DataFrame()

    # Recover which job each nonmatched skill belongs to
    job_indices = np.where(nonmatch_mask)[0]  # row index = job index

    cluster_labels = DBSCAN(eps=0.2, min_samples=2, metric="cosine").fit_predict(flat_embeddings)

    total_jobs = len(market_obj.job_id_by_index)
    df = (
        pd.DataFrame({"skill": flat_descriptions, "job_index": job_indices, "cluster": cluster_labels})
        .groupby("cluster")
        .agg(
            skill_variants=("skill", lambda x: list(set(x))),
            total_matches=("skill", "count"),
            unique_jobs=("job_index", "nunique")
        )
        .assign(job_coverage_pct=lambda df: (df["unique_jobs"] / total_jobs * 100).round(2))
        .sort_values("total_matches", ascending=False)
        .reset_index(drop=True)
    )
    return df[df["skill_variants"].apply(len) < 10] # Avoids poorly formed skill variants where a large number of non related items are grouped together


### Candidate Logic

In [137]:
class HardSkill(BaseModel):
    description: str = Field(
        description=(
            "Proficiency phrase + canonical skill name + optional context from the candidate's actual usage. "
            "Must start with one of: 'Familiar with', 'Working knowledge of', 'Experience with', "
            "'Proficient in', 'Expert-level mastery of'. "
            "Choose based on evidence in the resume: years of use, seniority of role, leadership signals. "
            "Examples: 'Proficient in Python for data pipeline development', "
            "'Expert-level mastery of Spark Streaming for large-scale ingestion', "
            "'Experience with AWS for cloud-based data infrastructure'."
        )
    )
    time_experience_months: Optional[float] = Field(
        description="Months of experience with this specific skill, inferred from role durations if not stated explicitly. Null if not determinable."
    )
    weight: Optional[float] = Field(
        description=(
            "How central this skill is to the candidate's professional identity, 0–1. "
            "Base on recency, frequency across roles, and seniority of use — not just mention count."
        )
    )

class SoftSkill(BaseModel):
    description: str = Field(
        description=(
            "Verb phrase describing an observable behavior, inferred from how the candidate describes their work. "
            "Format: '<verb>-ing <object> [<context>]'. "
            "Examples: 'Collaborating across engineering and product teams', "
            "'Communicating technical decisions to non-technical stakeholders'. "
            "Never use noun phrases like 'Stakeholder Communication'."
        )
    )
    weight: Optional[float] = Field(
        description="How consistently this trait appears across the candidate's experience, 0–1."
    )

class Position(BaseModel):
    name: str = Field(description="Name of the goal position or most experienced position")
    time_experience_months: Optional[float] = Field(description="Time experience in months identified for the held position. Set to 0 if not able to identify.")

class ProfessionalProfile(BaseModel):
    industries: List[str] = Field(description="Maximum of 2 matching LinkedIn industries list for the current goal job title, not the experience. Must be chosen from the list provided.")
    seniority: str = Field(description=f"Seniority level identifified for main goal position. Must be one of: {', '.join(SENIORITY_LEVELS)}.")
    position: Position
    hard_skills: List[HardSkill]
    soft_skills: List[SoftSkill]

_RESUME_SYSTEM_PROMPT = """
You are an expert technical recruiter building a structured skill profile from a candidate's resume.

## Step 1 — Read the full resume first
Identify all roles, their durations, and the technical environment of each before extracting anything.
Use this to infer proficiency levels and how central each skill is to the candidate's career.

## Step 2 — Proficiency inference for resume signals

| Resume signal                                              | Use                      |
|------------------------------------------------------------|--------------------------|
| Mentioned once, peripheral role, or certification only     | Familiar with            |
| Used in a supporting capacity or older/shorter role        | Working knowledge of     |
| Used across one or more roles with clear contribution      | Experience with          |
| Core tool across multiple roles, appears in achievements   | Proficient in            |
| Led adoption, designed systems around it, taught others    | Expert-level mastery of  |

## Step 3 — Hard skills
Format: `<proficiency phrase> <canonical name> [for <context from how the candidate used it>]`

Rules:
- Extract the broad category AND each specific tool separately.
- Context clause: use the candidate's actual application, not the tool's generic purpose.
  Bad  → `Experience with Kafka for message queuing`
  Good → `Experience with Kafka for real-time event streaming in financial transaction systems`
- Canonical names only: `Apache Spark`, `Python`, `AWS` — not too broad, not version-specific.
- Infer `time_experience_months` from overlapping role durations when not stated explicitly.
- Weight reflects career centrality: a tool used in every recent role outweighs one mentioned once.

## Step 4 — Soft skills
Infer from HOW the candidate describes their work, not just explicit claims.
"Led a team of 5" → `Leading engineering teams through delivery cycles`
"Reduced pipeline latency by 40%" → `Driving performance optimization through data-informed decisions`

Format: `<verb>-ing <object> [<context>]` — verb phrases only, never noun labels.

## Step 5 — Weight guidance

| Proficiency level           | Core to career | Supporting role | Peripheral |
|-----------------------------|----------------|-----------------|------------|
| Expert-level mastery of     | 0.90 – 1.0     | —               | —          |
| Proficient in               | 0.75 – 0.90    | 0.55 – 0.70     | —          |
| Experience with             | 0.55 – 0.75    | 0.35 – 0.55     | —          |
| Working knowledge of        | 0.30 – 0.55    | 0.20 – 0.35     | —          |
| Familiar with               | —              | 0.15 – 0.30     | 0.10 – 0.20|

All output must be in English, even if the resume is in another language.
{industries_list}
"""

def extract_professional_structured_data(text: str) -> dict:
    try:
        llm = init_chat_model(
            model=LLM_MODEL_NAME,
            model_provider=LLM_PROVIDER,
            temperature=LLM_TEMPERATURE,
            api_key=api_key,
        )
        system_prompt = _RESUME_SYSTEM_PROMPT.format(
            industries_list=f"Industries list to choose from for the goal position: {'|'.join(get_static_list_of_industries())}"
        )
        prompt = ChatPromptTemplate.from_messages([
            ("system", system_prompt),
            ("human", "{input}"),
        ])
        structured_llm = llm.with_structured_output(schema=ProfessionalProfile)
        chain = prompt | structured_llm
        response = chain.invoke({"input": text})
        return response.model_dump()
    except Exception as e:
        raise StructuredOutputParsingError(detail=str(e))
    

# Get resume from DB
RESUME_ID = "256effd5-6c42-4ac2-8818-f8ffd5e89aa4"  # REMOVE, must be a parameter
db = DatabaseManager()
resume_df = db.get_resume(RESUME_ID)
resume_id = resume_df['id'].iloc[0]
resume_text = resume_df['description'].iloc[0]
profile_dict = extract_professional_structured_data(resume_text)

client = genai.Client(api_key=api_key)
hard_skill_descriptions = [skill["description"] for skill in profile_dict["hard_skills"]]
soft_skill_descriptions = [skill["description"] for skill in profile_dict["soft_skills"]]
all_descriptions = hard_skill_descriptions + soft_skill_descriptions

embeddings = embed_column(client, all_descriptions) if all_descriptions else []

hard_records = []
for idx, skill in enumerate(profile_dict["hard_skills"]):
    hard_records.append({
        "id": str(uuid.uuid4()),
        "resume_id": resume_id,
        "skill_description": skill["description"],
        "weight": skill.get("weight", 0.5),
        "time_experience_months": skill.get("time_experience_months", 0),
        "embedding": embeddings[idx] if idx < len(embeddings) else None,
    })
candidate_hard_skills_df = pd.DataFrame(hard_records)

soft_records = []
for idx, skill in enumerate(profile_dict["soft_skills"]):
    soft_records.append({
        "id": str(uuid.uuid4()),
        "resume_id": resume_id,
        "skill_description": skill["description"],
        "weight": skill.get("weight", 0.5),
        "embedding": embeddings[len(hard_skill_descriptions) + idx] if len(hard_skill_descriptions) + idx < len(embeddings) else None,
    })
candidate_soft_skills_df = pd.DataFrame(soft_records)

print(f"Hard Skills ({len(candidate_hard_skills_df)} skills)")
print(f"Soft Skills ({len(candidate_soft_skills_df)} skills)")

candidate_industries = ["INFORMATION TECHNOLOGY AND SERVICES","DATA INFRASTRUCTURE AND ANALYTICS"]

# Add embeddings to pre-loaded job skills dataframes
job_hard_descriptions = hard_df["skill_description"].tolist()
job_soft_descriptions = soft_df["skill_description"].tolist()
job_all_descriptions = job_hard_descriptions + job_soft_descriptions

job_embeddings = embed_column(client, job_all_descriptions) if job_all_descriptions else []

hard_df_with_embeddings = hard_df.copy()
hard_df_with_embeddings["embedding"] = [
    job_embeddings[idx] if idx < len(job_embeddings) else None
    for idx in range(len(hard_df))
]

soft_df_with_embeddings = soft_df.copy()
soft_df_with_embeddings["embedding"] = [
    job_embeddings[len(job_hard_descriptions) + idx] if len(job_hard_descriptions) + idx < len(job_embeddings) else None
    for idx in range(len(soft_df))
]

# Build market objects directly from pre-loaded dataframes with embeddings
def build_market_from_preloaded(skills_df, skill_type):
    """Build a MarketSkillsMatrix-compatible object from pre-loaded dataframes."""
    market = MarketSkillsMatrix(database_manager=db, skill_type=skill_type, candidate_industries=candidate_industries)
    
    # Rebuild matrices using pre-loaded data instead of querying DB
    skills_data = skills_df.copy().sort_values("job_id")
    skills_data["index"] = skills_data.groupby("job_id").ngroup()
    max_skills_per_job = skills_data.groupby("index").size().max()
    
    market.string_matrix = build_padded_matrix(skills_data, "skill_description", max_skills_per_job, pad_value="")
    market.weight_matrix = build_padded_matrix(skills_data, "weight", max_skills_per_job, pad_value=0, dtype=np.float32)
    market.embedding_matrix = build_embedding_matrix(skills_data, max_skills_per_job)
    market.skills_count_by_index = skills_data["index"].value_counts().sort_index().tolist()
    market.job_id_by_index = skills_data[["index", "job_id"]].drop_duplicates()["job_id"].tolist()
    market._match_score_matrix = create_match_score_matrix(market.skills_count_by_index)
    market._weighted_match_matrix = np.zeros_like(market._match_score_matrix)
    
    return market

soft_market = build_market_from_preloaded(soft_df_with_embeddings, "soft")
hard_market = build_market_from_preloaded(hard_df_with_embeddings, "hard")

analyze_market(soft_market, candidate_soft_skills_df, "soft")
analyze_market(hard_market, candidate_hard_skills_df, "hard")

market_soft_skills_analysis = get_market_analysis_results(soft_market)
market_hard_skills_analysis = get_market_analysis_results(hard_market)
soft_market_analysis_df = build_analysis_display(soft_market, market_soft_skills_analysis)
hard_market_analysis_df = build_analysis_display(hard_market, market_hard_skills_analysis)
compliant_soft_skills_report = matches_display(soft_market, market_soft_skills_analysis)    
compliant_hard_skills_report = matches_display(hard_market, market_hard_skills_analysis) 

noncompliant_soft_skills_report = nonmatches_display(soft_market)
noncompliant_hard_skills_report = nonmatches_display(hard_market)

# API Endpoint 1 for Compliant Skill (soft and hard)
candidate_soft_compliance_breakdown = soft_market_analysis_df.sort_values("minimum_compliance_pct", ascending=False).to_dict(orient="records")
candidate_hard_compliance_breakdown = hard_market_analysis_df.sort_values("minimum_compliance_pct", ascending=False).to_dict(orient="records")

# API Endpoint 2: Compliant Skills (soft and hard)
compliant_skills_coverage = { 
    "soft_skills": compliant_soft_skills_report.to_dict(orient="records"),
    "hard_skills": compliant_hard_skills_report.to_dict(orient="records") 
}
# API Endpoint 3: Non Compliant Skills (soft and hard)
noncompliant_skills_coverage = { 
    "soft_skills": noncompliant_soft_skills_report.to_dict(orient="records"),
    "hard_skills": noncompliant_hard_skills_report.to_dict(orient="records") 
}
db.close_all()


Initializing connection pool...
Hard Skills (43 skills)
Soft Skills (11 skills)
Rate limited — waiting 10s (attempt 1/5)
Rate limited — waiting 56s (attempt 1/5)
Rate limited — waiting 56s (attempt 2/5)
Rate limited — waiting 55s (attempt 1/5)
Rate limited — waiting 54s (attempt 1/5)
Rate limited — waiting 54s (attempt 1/5)
Rate limited — waiting 20s (attempt 2/5)
Rate limited — waiting 10s (attempt 1/5)
Rate limited — waiting 10s (attempt 1/5)
Rate limited — waiting 10s (attempt 1/5)
Rate limited — waiting 20s (attempt 2/5)
Rate limited — waiting 20s (attempt 2/5)
Rate limited — waiting 10s (attempt 1/5)
Rate limited — waiting 10s (attempt 1/5)
Rate limited — waiting 10s (attempt 1/5)
Rate limited — waiting 10s (attempt 1/5)
Rate limited — waiting 20s (attempt 2/5)
Rate limited — waiting 40s (attempt 3/5)
Rate limited — waiting 40s (attempt 3/5)
Rate limited — waiting 10s (attempt 1/5)
Rate limited — waiting 6s (attempt 1/5)
Rate limited — waiting 5s (attempt 1/5)
Rate limited — waiti

## Display Results

In [138]:
candidate_soft_skills_df[["skill_description", "weight"]]

,skill_description,weight
0,Demonstrating ownership and accountability in ...,0.90
1,Exhibiting autonomy and initiative in problem-...,0.85
2,Maintaining focus and self-management in dynam...,0.80
3,Making sound technical decisions based on anal...,0.85
4,Sharing knowledge and expertise with team memb...,0.75
5,Producing clear and comprehensive technical do...,0.70
6,Translating complex business needs into action...,0.80
7,Collaborating effectively with cross-functiona...,0.85
8,Communicating technical concepts to both techn...,0.80
9,Driving innovation through creative problem-so...,0.70


In [139]:
candidate_hard_skills_df[["skill_description", "weight", "time_experience_months"]]

,skill_description,weight,time_experience_months
0,Expert-level mastery of Google Cloud Platform ...,0.95,39.0
1,Proficient in AWS for cloud-based data infrast...,0.80,27.0
2,Expert-level mastery of BigQuery for data ware...,0.95,39.0
3,Proficient in Apache Airflow for pipeline orch...,0.85,39.0
4,Proficient in Python for data pipeline develop...,0.85,39.0
5,Proficient in SQL for data manipulation and an...,0.85,39.0
6,Experience with Apache Beam for building high-...,0.60,12.0
7,Experience with Dataform for automated data pi...,0.60,12.0
8,Experience with Docker for containerization,0.60,27.0
9,Experience with Git for version control,0.70,39.0


In [140]:
pd.DataFrame(candidate_soft_compliance_breakdown)

,job_id,job_index,required_skills,matched_count,minimum_compliance_pct,matched_skills,insufficient_count,ideal_compliance_pct,insufficient_proficiency,nonmatched_count,nonmatched_skills
0,2064841470,40,7,7,100.00,"[Communicating insights clearly, Leading creat...",4,42.86,"[Driving continuous improvement, Enabling stra...",0,[]
1,2087612834,93,4,4,100.00,"[Contributing to architectural decisions, Ment...",2,50.00,"[Contributing to architectural decisions, Comm...",0,[]
2,2100658506,127,10,10,100.00,[Translating complex business problems into si...,2,80.00,[Acting as a technical reference across multip...,0,[]
3,2108492118,159,1,1,100.00,[Analyzing and solving problems for bug triage...,0,100.00,[],0,[]
4,2117217356,215,5,5,100.00,"[Documenting and sharing knowledge, Adapting t...",2,60.00,[Communicating effectively with technical and ...,0,[]
...,...,...,...,...,...,...,...,...,...,...,...
293,2050064882,18,38,4,10.53,"[Ensuring data traceability, Implementing data...",35,5.26,"[Automating data update flows, Designing data ...",34,"[Automating data update flows, Implementing da..."
294,2122980329,275,12,1,8.33,[Maintaining documentation],11,8.33,"[Managing CI/CD processes, Optimizing performa...",11,"[Managing CI/CD processes, Optimizing performa..."
295,2119589243,232,16,1,6.25,[Collaborating with a global team],15,6.25,"[Learning from top talent, Ensuring data gover...",15,"[Learning from top talent, Ensuring data gover..."
296,2119589251,233,17,1,5.88,[Collaborating with data architects],16,5.88,"[Defining data quality standards, Implementing...",16,"[Defining data quality standards, Implementing..."


In [141]:
pd.DataFrame(candidate_hard_compliance_breakdown)

,job_id,job_index,required_skills,matched_count,minimum_compliance_pct,matched_skills,insufficient_count,ideal_compliance_pct,insufficient_proficiency,nonmatched_count,nonmatched_skills
0,2035482953,5,25,25,100.00,"[Familiar with AWS S3, Familiar with AWS Lambd...",1,96.00,[Experience with Redshift],0,[]
1,2043631830,9,20,20,100.00,"[Working knowledge of Confluence, Experience w...",6,70.00,"[Expert-level mastery of Cloud Storage, Profic...",0,[]
2,2045540192,11,11,11,100.00,"[Proficient in Relational databases (Postgres,...",2,81.82,"[Proficient in Python, Proficient in English]",0,[]
3,2050064882,18,22,22,100.00,"[Proficient in Databricks platform, Proficient...",9,59.09,"[Proficient in data documentation, Proficient ...",0,[]
4,2050791840,19,18,18,100.00,"[Proficient in data engineering, Proficient in...",1,94.44,[Proficient in Python],0,[]
...,...,...,...,...,...,...,...,...,...,...,...
293,2100718408,131,13,6,46.15,[Proficient in Python for data manipulation an...,11,15.38,[Familiar with LLMs for document understanding...,7,[Familiar with LLMs for document understanding...
294,2062222085,34,31,14,45.16,"[Experience with power distribution systems, P...",22,29.03,[Experience with critical environment and indu...,17,[Experience with critical environment and indu...
295,2095208171,108,21,8,38.10,"[Experience with Agile/Scrum teams, Experience...",14,33.33,[Familiar with Codex for AI-assisted developme...,13,[Familiar with Codex for AI-assisted developme...
296,2112447938,179,12,4,33.33,[Familiar with GitHub Copilot [for developing ...,9,25.00,"[Familiar with Generative AI [for creating, te...",8,"[Familiar with Generative AI [for creating, te..."


In [142]:
compliant_soft_skills_report.sort_values("job_coverage_pct", ascending=False)

,skill_variants,total_matches,unique_jobs,job_coverage_pct
6,[Solving problems],54,18,6.04
14,[Communicating effectively],30,15,5.03
20,"[Communicating effectively in writing, Communi...",22,11,3.69
7,"[Handling technical challenges, Identifying is...",48,9,3.02
48,"[Working effectively in agile teams, Working i...",12,9,3.02
...,...,...,...,...
927,"[Influencing testing and commissioning teams, ...",2,1,0.34
926,[Enhancing operational excellence through know...,2,1,0.34
925,[Ensuring alignment with security practices],2,1,0.34
924,[Enabling strategic decision-making],2,1,0.34


In [143]:
compliant_hard_skills_report.sort_values("job_coverage_pct", ascending=False)

,skill_variants,total_matches,unique_jobs,job_coverage_pct
3,"[Experience with Metadata Management, Experien...",162,11,3.69
23,[Experience with RAG (Retrieval-Augmented Gene...,40,9,3.02
77,"[Experience with Medallion Architecture, Exper...",21,9,3.02
235,[Experience with accounting in finance/fintech...,9,7,2.35
2,"[Experience with microservices architecture, E...",191,7,2.35
...,...,...,...,...
402,[Proficient in Systems Availability Analysis],5,1,0.34
403,[Proficient in Power and Cooling Designs],5,1,0.34
404,[Proficient in VPC],5,1,0.34
405,[Experience with Celery],5,1,0.34


In [144]:
noncompliant_soft_skills_report.sort_values("job_coverage_pct", ascending=False)

,skill_variants,total_matches,unique_jobs,job_coverage_pct
3,"[Mentoring junior engineers, Being a role mode...",15,13,4.36
4,"[Prioritizing tasks effectively, Prioritizing ...",13,11,3.69
7,"[Attention to detail, Having forensic attentio...",10,10,3.36
8,"[Optimizing resource utilization, Optimizing c...",10,10,3.36
5,"[Verbal communication in English, Communicatin...",12,9,3.02
...,...,...,...,...
210,[Building automation frameworks for cost trans...,2,1,0.34
208,[Integrating with third-party platforms at sca...,2,1,0.34
212,"[Optimizing Databricks cost, Optimizing Databr...",2,1,0.34
215,"[Committing to self-improvement, Committing to...",2,1,0.34


In [145]:
noncompliant_hard_skills_report.sort_values("job_coverage_pct", ascending=False)

,skill_variants,total_matches,unique_jobs,job_coverage_pct
3,"[Familiar with dbt, Working knowledge of dbt i...",13,13,4.36
4,[Familiar with Site Reliability Engineering (S...,8,6,2.01
6,"[Working knowledge of Kafka, Familiar with Kafka]",6,6,2.01
15,"[Familiar with Kinesis, Experience with Kinesis]",4,4,1.34
7,"[Familiar with open source projects, Familiar ...",6,3,1.01
...,...,...,...,...
54,"[Working knowledge of HDInsight, Working knowl...",2,1,0.34
60,[Experience with Large Language Models (LLMs) ...,2,1,0.34
63,[Experience with Power Planning in Data Center...,2,1,0.34
64,"[Experience with L5 Commissioning Tests, Exper...",2,1,0.34
